In [1]:
import pandas as pd
import sys
from pathlib import Path

sys.path.insert(0, str(Path('.').resolve()))

from model.modular import Modular
from model.solution import Solution
from model.simulated_annealing import SimulatedAnnealing
from utils import plot_linecard_heatmaps

csv_path = 'database/cisco_small.csv'
requirement = pd.DataFrame([{
    'code': 'requirement',
    '100': 4,
    '40': 0,
    '25': 33,
    '10': 0,
}])

df = pd.read_csv(csv_path)
modules_df = df[df['type'] == 'modular']
linecards_df = df[df['type'] == 'linecard']

print(f"{len(modules_df)} module rows")
print(f"{linecards_df['code'].nunique()} unique linecards")
print(f"available modules: {modules_df['code'].unique().tolist()}")


1 module rows
2 unique linecards
available modules: ['9516']


In [2]:
results = {}
errors = {}
module_codes = modules_df['code'].unique()

for module_code in module_codes:
    try:
        module_data = df[df['code'] == module_code]
        module_family = module_data['family'].iloc[0]
        linecards_for_family = df[(df['type'] == 'linecard') & (df['family'] == module_family)]
        combined_data = pd.concat([module_data, linecards_for_family], ignore_index=True)

        modular = Modular(combined_data)
        solution = Solution(modular, requirement)
        result = solution.solve(heuristic="H2")

        if result is not None and not result.empty:
            results[module_code] = result
            print(f"✓ {module_code}: Found solution with {len(result)} linecards")
        else:
            errors[module_code] = "Requirement cannot be satisfied within maxmodules limit"
            print(f"✗ {module_code}: {errors[module_code]}")
    except Exception as e:
        errors[module_code] = str(e)
        print(f"✗ {module_code}: {str(e)[:100]}")

print(f"{len(results)} successful, {len(errors)} failed")

✓ 9516: Found solution with 2 linecards
1 successful, 0 failed


In [3]:
if results:
    print("SUCCESSFUL CONFIGURATIONS")

    for module_code, solution in results.items():
        print(module_code)

        speed_columns = [col for col in solution.columns if col not in ['code', 'value']]
        speed_columns_sorted = sorted(
            speed_columns,
            key=lambda x: float(x) if x not in ['code', 'value'] else 0,
            reverse=True
        )

        # Add cost column based on linecard costs
        solution = solution.copy()
        solution['cost'] = solution['code'].apply(
            lambda code: next((lc.cost for lc in modular.linecards if lc.code == code), 0)
        )

        display_cols = ['code'] + speed_columns_sorted + ['value'] + ["cost"]

        display_solution = solution[display_cols].copy()
        display_solution = display_solution.fillna(0)
        for col in speed_columns_sorted + ['value']:
            display_solution[col] = display_solution[col].astype(int)

        print(display_solution.to_string(index=False))

        total_value = int(solution['value'].sum())
        print(f"\n✓ Total value (throughput*ports): {total_value}")
else:
    print("No successful configurations found.")


SUCCESSFUL CONFIGURATIONS
9516
          code  100  40  25  10  1  value   cost
N9K-X96136YC-R    4   0  32   0  0   1200 3000.0
N9K-X96136YC-R    0   0   1   0  0     25 3000.0

✓ Total value (throughput*ports): 1225


In [4]:
sa_results = {}

for module_code in module_codes:
    try:
        module_data = df[df['code'] == module_code]
        module_family = module_data['family'].iloc[0]
        linecards_for_family = df[(df['type'] == 'linecard') & (df['family'] == module_family)]
        combined_data = pd.concat([module_data, linecards_for_family], ignore_index=True)

        modular = Modular(combined_data)

        # Initialize SA with Solution class and parameters
        sa = SimulatedAnnealing(
            l=0.1,
            t=100.0,
            state=Solution,
            modular=modular,
            req=requirement
        )

        best_solution = sa.run()
        best_score = best_solution.score()
        best_cost = -best_score if best_score != float("-inf") else float("inf")
        try:
            best_solution_df = best_solution.solve(heuristic="H2")
        except Exception:
            best_solution_df = pd.DataFrame()

        sa_results[module_code] = {
            'solution': best_solution,
            'solution_df': best_solution_df,
            'cost': best_cost,
            'history': sa.history_arrays(),
            'attempt_history': sa.attempt_history_arrays()
        }

        if best_cost == float("inf"):
            print(f"✗ {module_code}: SA produced no valid solution")
        else:
            print(f"✓ {module_code}: Best cost = ${best_cost:.2f}")

    except Exception as e:
        print(f"✗ {module_code}: SA failed - {str(e)[:100]}")


✓ 9516: Best cost = $3200.00


In [5]:
if sa_results:
    for module_code, result in sa_results.items():
        best_sol = result['solution']
        cost = result['cost']

        print(f"\n{'='*60}")
        print(f"Module: {module_code}")
        print(f"Total Cost: ${cost:.2f}")
        print(f"{'='*60}")

        solution = result.get('solution_df', pd.DataFrame())
        if solution is not None and not solution.empty:
            # Get and sort speed columns
            speed_columns = [col for col in solution.columns if col not in ['code', 'value']]
            speed_columns_sorted = sorted(
                speed_columns,
                key=lambda x: float(x) if x not in ['code', 'value'] else 0,
                reverse=True
            )

            # Add cost column based on linecard costs
            solution_with_cost = solution.copy()
            solution_with_cost['cost'] = solution_with_cost['code'].apply(
                lambda code: next((lc.cost for lc in best_sol.modular.linecards if lc.code == code), 0)
            )

            display_cols = ['code'] + speed_columns_sorted + ['value', 'cost']

            display_solution = solution_with_cost[display_cols].copy()
            display_solution = display_solution.fillna(0)
            for col in speed_columns_sorted + ['value']:
                display_solution[col] = display_solution[col].astype(int)

            print("\nSelected Linecards:")
            print(display_solution.to_string(index=False))

            total_value = int(solution['value'].sum())
            print(f"\nTotal value (throughput*ports): {total_value}")
            print(f"Total cost: ${solution_with_cost['cost'].sum():.2f}")

        # Show convergence info
        step_history, scores, temps = result['history']
        print(f"\nConvergence Info:")
        print(f"   Iterations: {len(scores)}")
        if scores:
            print(f"   Initial cost: ${-scores[0]:.2f}")
            print(f"   Final cost: ${-scores[-1]:.2f}")

        try:
            attempt_step_history, attempt_scores, _ = result.get('attempt_history', ([], [], []))
            best_indices = []
            best_score = None
            for idx, score in enumerate(attempt_scores):
                if best_score is None or score > best_score:
                    best_score = score
                    best_indices.append(idx)

            plot_linecard_heatmaps(
                attempt_step_history if attempt_step_history else step_history,
                max_attempts=50,
                title_prefix=f"{module_code} Step",
                best_attempt_indices=best_indices
            )
        except Exception as e:
            print(f"Heatmap skipped: {str(e)[:100]}")
else:
    print("No SA results available.")



Module: 9516
Total Cost: $3200.00

Selected Linecards:
           code  100  40  25  10  1  value   cost
N9K-X97160YC-EX    4   0  33   0  0   1225 3200.0

Total value (throughput*ports): 1225
Total cost: $3200.00

Convergence Info:
   Iterations: 117
   Initial cost: $6000.00
   Final cost: $3200.00
